# Este notebook analiza los **30 SBOM (Software Bill of Materials)** generados por **Syft** para los repositorios de TanStack.

Los SBOM utilizan **CycloneDX** como formato de intercambio y describen los componentes detectados en cada repositorio.

El análisis se concentra en:

- cantidad de componentes por repositorio;
- componentes y paquetes únicos;
- tipos de componentes y ecosistemas;
- dependencias compartidas entre repositorios;
- diversidad de versiones;
- componentes con versión `UNKNOWN`;
- cobertura de identificadores PURL y CPE;
- información de licencias;
- relaciones de dependencia incluidas en CycloneDX;
- concentración y reutilización de dependencias;
- resultados estructurados para el Visualizer.



## 1. Importaciones

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown


## 2. Localización de los archivos SBOM

In [ ]:
import os

_env_sbom_dir = os.environ.get("ANALYZER_SBOM_DIR")

candidate_directories = []
if _env_sbom_dir:
    candidate_directories.append(Path(_env_sbom_dir))

candidate_directories += [
    Path("../../results-sboms"),
    Path("../../resultados-sbom"),
    Path("../results-sboms"),
    Path("../resultados-sbom"),
    Path("results-sboms"),
    Path("resultados-sbom"),
    Path("."),
]

SBOM_DIR = next(
    (
        directory
        for directory in candidate_directories
        if directory.exists() and list(directory.glob("*.cdx.json"))
    ),
    None
)

if SBOM_DIR is None:
    raise FileNotFoundError(
        "No se encontró una carpeta con archivos *.cdx.json. "
        "Define ANALYZER_SBOM_DIR o ajusta candidate_directories."
    )

sbom_files = sorted(SBOM_DIR.glob("*.cdx.json"))

print("Carpeta utilizada:", SBOM_DIR.resolve())
print("Cantidad de SBOM encontrados:", len(sbom_files))

for path in sbom_files:
    print("-", path.name)

Para esta ejecución de TanStack se esperan **30 archivos SBOM**. El notebook no se detiene si la cantidad cambia, pero muestra una advertencia para mantener la trazabilidad de la ejecución.

## 3. Validación de formato y metadatos

In [ ]:
metadata_rows = []

for path in sbom_files:
    with open(path, "r", encoding="utf-8") as f:
        sbom = json.load(f)

    tools = (
        sbom.get("metadata", {})
        .get("tools", {})
        .get("components", [])
        or []
    )

    syft_tool = next(
        (
            tool for tool in tools
            if str(tool.get("name", "")).lower() == "syft"
        ),
        {}
    )

    metadata_rows.append({
        "repository": path.name.replace(".cdx.json", ""),
        "bom_format": sbom.get("bomFormat"),
        "spec_version": sbom.get("specVersion"),
        "serial_number": sbom.get("serialNumber"),
        "timestamp": sbom.get("metadata", {}).get("timestamp"),
        "syft_version": syft_tool.get("version"),
        "raw_components": len(sbom.get("components", []) or []),
        "dependency_nodes": len(sbom.get("dependencies", []) or [])
    })

df_metadata = pd.DataFrame(metadata_rows).sort_values("repository")

df_metadata


In [ ]:
print("SBOM procesados:", len(df_metadata))
print("Formatos:", df_metadata["bom_format"].dropna().unique())
print("Versiones CycloneDX:", df_metadata["spec_version"].dropna().unique())
print("Versiones Syft:", df_metadata["syft_version"].dropna().unique())

invalid_format = df_metadata[
    df_metadata["bom_format"].fillna("") != "CycloneDX"
]

if invalid_format.empty:
    print("\nTodos los archivos declaran formato CycloneDX.")
else:
    print("\nADVERTENCIA: existen archivos que no declaran CycloneDX:")
    display(invalid_format)

if len(df_metadata) == 30:
    print("Se encontraron los 30 SBOM esperados.")
else:
    print(
        f"ADVERTENCIA: se esperaban 30 SBOM y se encontraron {len(df_metadata)}."
    )


Los archivos de esta ejecución deben ser comparables entre sí. También es útil verificar que fueron generados con la misma versión de Syft y la misma versión de CycloneDX.

## 4. Extracción de componentes y relaciones

In [ ]:
component_rows = []
dependency_rows = []

def extract_licenses(component):
    values = []

    for item in component.get("licenses", []) or []:
        if not isinstance(item, dict):
            continue

        license_obj = item.get("license")

        if isinstance(license_obj, dict):
            value = license_obj.get("id") or license_obj.get("name")
            if value:
                values.append(value)

        if item.get("expression"):
            values.append(item["expression"])

    return " | ".join(sorted(set(values)))


def extract_ecosystem(purl):
    if not isinstance(purl, str) or not purl.startswith("pkg:"):
        return None

    return purl[4:].split("/", 1)[0]


for path in sbom_files:
    repository = path.name.replace(".cdx.json", "")

    with open(path, "r", encoding="utf-8") as f:
        sbom = json.load(f)

    for component in sbom.get("components", []) or []:
        properties_list = component.get("properties", []) or []

        properties = {
            item.get("name"): item.get("value")
            for item in properties_list
            if isinstance(item, dict) and item.get("name")
        }

        locations = [
            item.get("value")
            for item in properties_list
            if (
                isinstance(item, dict)
                and str(item.get("name", "")).startswith("syft:location:")
                and str(item.get("name", "")).endswith(":path")
                and item.get("value")
            )
        ]

        purl = component.get("purl")

        component_rows.append({
            "repository": repository,
            "component_type": component.get("type"),
            "name": component.get("name"),
            "version": component.get("version"),
            "purl": purl,
            "ecosystem": extract_ecosystem(purl),
            "cpe": component.get("cpe"),
            "licenses": extract_licenses(component),
            "found_by": properties.get("syft:package:foundBy"),
            "language": properties.get("syft:package:language"),
            "package_type": properties.get("syft:package:type"),
            "metadata_type": properties.get("syft:package:metadataType"),
            "location": locations[0] if locations else None,
            "bom_ref": component.get("bom-ref")
        })

    for dependency in sbom.get("dependencies", []) or []:
        source = dependency.get("ref")

        for target in dependency.get("dependsOn", []) or []:
            dependency_rows.append({
                "repository": repository,
                "source_ref": source,
                "target_ref": target
            })

component_columns = [
    "repository", "component_type", "name", "version", "purl",
    "ecosystem", "cpe", "licenses", "found_by", "language",
    "package_type", "metadata_type", "location", "bom_ref"
]

dependency_columns = [
    "repository", "source_ref", "target_ref"
]

df_components_raw = pd.DataFrame(
    component_rows,
    columns=component_columns
)

df_dependency_edges = pd.DataFrame(
    dependency_rows,
    columns=dependency_columns
)

print("Apariciones de componentes:", len(df_components_raw))
print("Relaciones de dependencia:", len(df_dependency_edges))

df_components_raw.head()


Syft puede registrar un mismo paquete más de una vez si aparece en diferentes archivos de bloqueo o ubicaciones. Por eso se conservarán dos vistas:

- `df_components_raw`: todas las apariciones detectadas;
- `df_components`: componentes únicos dentro de cada repositorio.


## 5. Deduplicación de componentes

In [ ]:
df_components_raw["component_identity"] = (
    df_components_raw["purl"]
    .fillna(
        df_components_raw["name"].fillna("")
        + "@"
        + df_components_raw["version"].fillna("UNKNOWN")
        + "#"
        + df_components_raw["component_type"].fillna("")
    )
)

df_components = (
    df_components_raw
    .drop_duplicates(
        subset=["repository", "component_identity"]
    )
    .copy()
)

print("Apariciones originales:", len(df_components_raw))
print("Componentes únicos por repositorio:", len(df_components))
print(
    "Apariciones duplicadas eliminadas:",
    len(df_components_raw) - len(df_components)
)


## 6. Componentes por repositorio

In [ ]:
repo_summary = df_components.groupby("repository").agg(
    unique_components=("component_identity", "nunique"),
    unique_component_names=("name", "nunique"),
    npm_components=("package_type", lambda s: (s == "npm").sum()),
    github_action_components=(
        "package_type",
        lambda s: s.isin(
            ["github-action", "github-action-workflow"]
        ).sum()
    ),
    unknown_versions=(
        "version",
        lambda s: s.fillna("UNKNOWN").eq("UNKNOWN").sum()
    ),
    with_purl=("purl", lambda s: s.notna().sum()),
    with_cpe=("cpe", lambda s: s.notna().sum()),
    with_license=(
        "licenses",
        lambda s: s.fillna("").ne("").sum()
    )
)

raw_counts = (
    df_components_raw.groupby("repository")
    .size()
    .rename("raw_component_occurrences")
)

repo_summary = repo_summary.join(raw_counts)

repo_summary["purl_percentage"] = (
    repo_summary["with_purl"]
    / repo_summary["unique_components"]
    * 100
).replace([np.inf, -np.inf], np.nan).fillna(0).round(2)

repo_summary["cpe_percentage"] = (
    repo_summary["with_cpe"]
    / repo_summary["unique_components"]
    * 100
).replace([np.inf, -np.inf], np.nan).fillna(0).round(2)

repo_summary["license_percentage"] = (
    repo_summary["with_license"]
    / repo_summary["unique_components"]
    * 100
).replace([np.inf, -np.inf], np.nan).fillna(0).round(2)

# Agrega repositorios con SBOM vacío para que no desaparezcan del resumen.
all_repositories = pd.Index(
    df_metadata["repository"],
    name="repository"
)

repo_summary = (
    repo_summary
    .reindex(all_repositories)
    .fillna(0)
)

integer_columns = [
    "unique_components",
    "unique_component_names",
    "npm_components",
    "github_action_components",
    "unknown_versions",
    "with_purl",
    "with_cpe",
    "with_license",
    "raw_component_occurrences"
]

repo_summary[integer_columns] = (
    repo_summary[integer_columns].astype(int)
)

repo_summary = repo_summary.sort_values(
    "unique_components",
    ascending=False
)

repo_summary


In [ ]:
repo_summary["unique_components"].head(15).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("Top 15 repositorios con más componentes únicos")
plt.xlabel("Cantidad de componentes únicos")
plt.ylabel("Repositorio")
plt.tight_layout()
plt.show()


## 7. SBOM vacíos

In [ ]:
empty_sboms = df_metadata[
    df_metadata["raw_components"] == 0
][["repository", "raw_components"]]

empty_sboms


In [ ]:
if empty_sboms.empty:
    display(Markdown("No se identificaron SBOM vacíos."))
else:
    names = ", ".join(
        f"`{repo}`"
        for repo in empty_sboms["repository"]
    )

    display(Markdown(
        f"Se identificaron **{len(empty_sboms)} SBOM vacíos**: {names}. "
        "Un SBOM vacío indica que Syft no registró componentes para ese repositorio; "
        "no significa automáticamente que el repositorio sea seguro ni que carezca de código."
    ))


## 8. Tipos de componentes

In [ ]:
component_type_counts = (
    df_components["component_type"]
    .fillna("sin_clasificar")
    .value_counts()
)

component_type_counts


In [ ]:
package_type_counts = (
    df_components["package_type"]
    .fillna("sin_clasificar")
    .value_counts()
)

package_type_counts


In [ ]:
package_type_counts.head(10).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Tipos de paquetes detectados por Syft")
plt.xlabel("Cantidad de componentes")
plt.ylabel("Tipo de paquete")
plt.tight_layout()
plt.show()


Los SBOM no contienen únicamente paquetes npm. Syft también puede detectar elementos como GitHub Actions y workflows. Separarlos evita interpretar erróneamente todos los componentes como dependencias JavaScript.

## 9. Ecosistemas identificados mediante PURL

In [ ]:
ecosystem_counts = (
    df_components["ecosystem"]
    .fillna("sin_purl")
    .value_counts()
)

ecosystem_counts


In [ ]:
ecosystem_counts.head(10).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Ecosistemas identificados mediante PURL")
plt.xlabel("Cantidad de componentes")
plt.ylabel("Ecosistema")
plt.tight_layout()
plt.show()


## 10. Paquetes npm compartidos entre repositorios

In [ ]:
df_npm = df_components[
    df_components["package_type"] == "npm"
].copy()

shared_packages = (
    df_npm.groupby("name")
    .agg(
        repositories=("repository", "nunique"),
        distinct_versions=("version", "nunique"),
        occurrences=("repository", "size")
    )
    .sort_values(
        ["repositories", "occurrences"],
        ascending=False
    )
)

shared_packages.head(25)


In [ ]:
shared_packages["repositories"].head(20).sort_values().plot(
    kind="barh",
    figsize=(10, 8)
)

plt.title("Paquetes npm presentes en más repositorios")
plt.xlabel("Cantidad de repositorios")
plt.ylabel("Paquete")
plt.tight_layout()
plt.show()


La cantidad de repositorios en que aparece un paquete permite medir su **reutilización organizacional**. Una dependencia muy compartida tiene mayor alcance potencial dentro de la organización si posteriormente Grype identifica una vulnerabilidad asociada a ella.

## 11. Cobertura organizacional de paquetes

In [ ]:
repository_count = df_metadata["repository"].nunique()

package_coverage = shared_packages.copy()

package_coverage["repository_percentage"] = (
    package_coverage["repositories"]
    / repository_count
    * 100
).round(2)

package_coverage.head(25)


In [ ]:
widely_shared = package_coverage[
    package_coverage["repository_percentage"] >= 50
]

print(
    "Paquetes npm presentes en al menos el 50% de los repositorios:",
    len(widely_shared)
)

widely_shared.head(30)


## 12. Diversidad de versiones

In [ ]:
version_diversity = (
    df_npm[
        df_npm["version"].notna()
        & df_npm["version"].ne("UNKNOWN")
    ]
    .groupby("name")
    .agg(
        distinct_versions=("version", "nunique"),
        repositories=("repository", "nunique")
    )
    .query("distinct_versions > 1")
    .sort_values(
        ["distinct_versions", "repositories"],
        ascending=False
    )
)

version_diversity.head(25)


In [ ]:
if version_diversity.empty:
    display(Markdown("No se identificaron paquetes npm con más de una versión."))
else:
    version_diversity["distinct_versions"].head(20).sort_values().plot(
        kind="barh",
        figsize=(10, 8)
    )

    plt.title("Paquetes npm con mayor diversidad de versiones")
    plt.xlabel("Cantidad de versiones distintas")
    plt.ylabel("Paquete")
    plt.tight_layout()
    plt.show()


La diversidad de versiones no equivale a una vulnerabilidad. Sin embargo, revela heterogeneidad en la gestión de dependencias y puede ayudar a explicar por qué una misma vulnerabilidad Grype aparece de forma diferente entre repositorios.

## 13. Detalle de versiones por paquete

In [ ]:
package_versions = (
    df_npm[
        df_npm["version"].notna()
    ]
    .groupby("name")["version"]
    .apply(lambda s: sorted(set(s)))
    .rename("versions")
    .to_frame()
)

package_versions["version_count"] = (
    package_versions["versions"].apply(len)
)

package_versions.sort_values(
    "version_count",
    ascending=False
).head(20)


## 14. Componentes con versión desconocida

In [ ]:
unknown_version_components = df_components[
    df_components["version"]
    .fillna("UNKNOWN")
    .eq("UNKNOWN")
].copy()

print(
    "Componentes con versión UNKNOWN:",
    len(unknown_version_components)
)

unknown_version_components[
    [
        "repository",
        "name",
        "component_type",
        "package_type",
        "location"
    ]
].head(30)


In [ ]:
unknown_by_type = (
    unknown_version_components["package_type"]
    .fillna("sin_clasificar")
    .value_counts()
)

unknown_by_type


In [ ]:
unknown_by_repo = (
    unknown_version_components["repository"]
    .value_counts()
)

unknown_by_repo.head(15)


Los componentes con versión `UNKNOWN` suelen corresponder a acciones o workflows locales y otros componentes para los que no existe una versión convencional detectable. No deben tratarse automáticamente como paquetes npm con información faltante.

## 15. Cobertura de PURL y CPE

In [ ]:
purl_count = int(df_components["purl"].notna().sum())
cpe_count = int(df_components["cpe"].notna().sum())
component_count = len(df_components)

purl_percentage = (
    purl_count / component_count * 100
    if component_count else 0
)

cpe_percentage = (
    cpe_count / component_count * 100
    if component_count else 0
)

identifier_coverage = pd.DataFrame({
    "identificador": ["PURL", "CPE"],
    "con_identificador": [purl_count, cpe_count],
    "porcentaje": [
        round(purl_percentage, 2),
        round(cpe_percentage, 2)
    ]
})

identifier_coverage


PURL es especialmente útil para identificar un paquete por ecosistema, nombre y versión, mientras que CPE proporciona otra forma estandarizada de identificación. La disponibilidad de estos identificadores facilita el cruce con herramientas como Grype.

## 16. Información de licencias

In [ ]:
license_mask = (
    df_components["licenses"]
    .fillna("")
    .ne("")
)

license_count = int(license_mask.sum())

license_percentage = (
    license_count / len(df_components) * 100
    if len(df_components) else 0
)

print(
    f"Componentes con información de licencia: "
    f"{license_count} ({license_percentage:.2f}%)"
)


In [ ]:
license_counts = (
    df_components.loc[license_mask, "licenses"]
    .value_counts()
)

license_counts.head(20)


In [ ]:
license_counts.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 7)
)

plt.title("Licencias más frecuentes declaradas en los SBOM")
plt.xlabel("Cantidad de componentes")
plt.ylabel("Licencia")
plt.tight_layout()
plt.show()


La ausencia de licencia en el SBOM no significa necesariamente que un componente carezca de licencia; indica que Syft no incorporó esa información para esa detección.

## 17. Relaciones de dependencia CycloneDX

In [ ]:
if df_dependency_edges.empty:
    dependency_summary = pd.DataFrame(
        columns=[
            "dependency_edges",
            "dependency_sources",
            "dependency_targets"
        ]
    )
else:
    dependency_summary = (
        df_dependency_edges.groupby("repository")
        .agg(
            dependency_edges=("target_ref", "size"),
            dependency_sources=("source_ref", "nunique"),
            dependency_targets=("target_ref", "nunique")
        )
        .sort_values(
            "dependency_edges",
            ascending=False
        )
    )

dependency_summary.head(15)


In [ ]:
if not dependency_summary.empty:
    dependency_summary["dependency_edges"].head(15).sort_values().plot(
        kind="barh",
        figsize=(10, 7)
    )

    plt.title("Repositorios con más relaciones de dependencia")
    plt.xlabel("Cantidad de relaciones")
    plt.ylabel("Repositorio")
    plt.tight_layout()
    plt.show()


Las relaciones CycloneDX permiten aproximarse a la complejidad del grafo de dependencias. Un mayor número de relaciones no implica automáticamente mayor riesgo de seguridad.

## 18. Componentes exclusivos y compartidos

In [ ]:
component_repository_frequency = (
    df_components.groupby(
        ["package_type", "name"],
        dropna=False
    )["repository"]
    .nunique()
    .rename("repositories")
    .reset_index()
)

exclusive_components = component_repository_frequency[
    component_repository_frequency["repositories"] == 1
]

shared_components = component_repository_frequency[
    component_repository_frequency["repositories"] > 1
]

print(
    "Nombres de componentes exclusivos de un repositorio:",
    len(exclusive_components)
)

print(
    "Nombres de componentes compartidos entre 2 o más repositorios:",
    len(shared_components)
)


## 19. Concentración de componentes

In [ ]:
component_concentration = (
    repo_summary[["unique_components"]]
    .sort_values(
        "unique_components",
        ascending=False
    )
    .copy()
)

total_repo_components = (
    component_concentration["unique_components"].sum()
)

component_concentration["percentage"] = (
    component_concentration["unique_components"]
    / total_repo_components
    * 100
).round(2)

component_concentration["cumulative_percentage"] = (
    component_concentration["percentage"]
    .cumsum()
    .round(2)
)

component_concentration.head(15)


In [ ]:
top3_component_share = (
    component_concentration.head(3)["unique_components"].sum()
    / total_repo_components
    * 100
    if total_repo_components else 0
)

top5_component_share = (
    component_concentration.head(5)["unique_components"].sum()
    / total_repo_components
    * 100
    if total_repo_components else 0
)

print(f"Top 3 repositorios: {top3_component_share:.2f}%")
print(f"Top 5 repositorios: {top5_component_share:.2f}%")


## 20. Comparación con `results.json` 

In [ ]:
_env_results = os.environ.get("ANALYZER_RESULTS_JSON")

results_candidates = ([Path(_env_results)] if _env_results else []) + [
    Path("../../results.json"),
    Path("../results.json"),
    Path("results.json"),
]

RESULTS_FILE = next(
    (path for path in results_candidates if path.exists()),
    None
)

if RESULTS_FILE is None:
    print(
        "No se encontró results.json. "
        "Se omite la validación cruzada."
    )
else:
    with open(RESULTS_FILE, "r", encoding="utf-8") as f:
        results_data = json.load(f)

    results_sbom_rows = []

    for repo in results_data.get("repositories", []):
        sbom_info = repo.get("sbom") or {}

        results_sbom_rows.append({
            "repository": repo.get("name"),
            "miner_sbom_status": sbom_info.get("status"),
            "miner_component_count": sbom_info.get("component_count"),
            "miner_sbom_path": sbom_info.get("path")
        })

    df_results_sbom = pd.DataFrame(results_sbom_rows)

    df_crosscheck = (
        df_metadata[
            ["repository", "raw_components"]
        ]
        .merge(
            df_results_sbom,
            on="repository",
            how="outer"
        )
    )

    df_crosscheck["component_count_matches"] = (
        df_crosscheck["raw_components"]
        == df_crosscheck["miner_component_count"]
    )

    display(df_crosscheck)

    mismatches = df_crosscheck[
        df_crosscheck["component_count_matches"] == False
    ]

    print(
        "Repositorios con diferencia entre SBOM y results.json:",
        len(mismatches)
    )


Esta comparación verifica que la cantidad bruta de componentes leída directamente desde cada SBOM coincida con la cantidad registrada por el Miner en `results.json`. Esto mejora la trazabilidad entre componentes.

## 21. Tabla final por repositorio

In [ ]:
final_repo_summary = repo_summary.join(
    dependency_summary,
    how="left"
)

for column in [
    "dependency_edges",
    "dependency_sources",
    "dependency_targets"
]:
    if column in final_repo_summary.columns:
        final_repo_summary[column] = (
            final_repo_summary[column]
            .fillna(0)
            .astype(int)
        )

final_repo_summary


## 22. Observaciones automáticas

In [ ]:
total_raw_components = len(df_components_raw)
total_unique_components = len(df_components)
duplicate_occurrences = (
    total_raw_components - total_unique_components
)

top_repo = (
    repo_summary.index[0]
    if not repo_summary.empty
    else "N/A"
)

top_repo_components = (
    int(repo_summary.iloc[0]["unique_components"])
    if not repo_summary.empty
    else 0
)

top_shared_package = (
    shared_packages.index[0]
    if not shared_packages.empty
    else "N/A"
)

top_shared_repositories = (
    int(shared_packages.iloc[0]["repositories"])
    if not shared_packages.empty
    else 0
)

empty_count = len(empty_sboms)

npm_count = int(
    (df_components["package_type"] == "npm").sum()
)

action_count = int(
    df_components["package_type"]
    .isin(["github-action", "github-action-workflow"])
    .sum()
)

summary_text = (
    "### Principales resultados\n\n"
    f"- Se procesaron **{len(df_metadata)} SBOM**.\n"
    f"- Se encontraron **{total_raw_components:,} apariciones de componentes** antes de deduplicar.\n"
    f"- Después de deduplicar por repositorio quedaron **{total_unique_components:,} componentes únicos por repositorio**.\n"
    f"- Se eliminaron **{duplicate_occurrences:,} apariciones repetidas** para evitar inflar comparaciones.\n"
    f"- Se identificaron **{npm_count:,} componentes npm** y **{action_count:,} componentes asociados a GitHub Actions/workflows**.\n"
    f"- El repositorio con más componentes únicos es **`{top_repo}`**, con **{top_repo_components:,}**.\n"
    f"- El paquete npm con mayor presencia entre repositorios es **`{top_shared_package}`**, presente en **{top_shared_repositories} repositorios**.\n"
    f"- Se identificaron **{empty_count} SBOM vacíos**.\n"
    f"- La cobertura global de PURL es **{purl_percentage:.2f}%**.\n"
    f"- La cobertura global de CPE es **{cpe_percentage:.2f}%**.\n"
    f"- La cobertura de información de licencias es **{license_percentage:.2f}%**.\n"
    f"- Los 3 repositorios con más componentes concentran **{top3_component_share:.2f}%** de los componentes contabilizados por repositorio.\n\n"
    "Estos resultados describen la **composición del software** y deben complementarse con Grype para determinar qué dependencias presentan vulnerabilidades conocidas."
)

display(Markdown(summary_text))


## 23. Exportación para el Visualizer

In [ ]:
import os

OUTPUT_DIR = Path(os.environ.get("ANALYZER_OUTPUT_DIR", "../output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_metadata.to_csv(
    OUTPUT_DIR / "sbom_metadata.csv",
    index=False
)

df_components.to_csv(
    OUTPUT_DIR / "sbom_components.csv",
    index=False
)

final_repo_summary.reset_index().to_csv(
    OUTPUT_DIR / "sbom_repository_summary.csv",
    index=False
)

package_coverage.reset_index().to_csv(
    OUTPUT_DIR / "sbom_shared_packages.csv",
    index=False
)

version_diversity.reset_index().to_csv(
    OUTPUT_DIR / "sbom_version_diversity.csv",
    index=False
)

unknown_version_components.to_csv(
    OUTPUT_DIR / "sbom_unknown_versions.csv",
    index=False
)

df_dependency_edges.to_csv(
    OUTPUT_DIR / "sbom_dependency_edges.csv",
    index=False
)

component_concentration.reset_index().to_csv(
    OUTPUT_DIR / "sbom_component_concentration.csv",
    index=False
)

final_repo_summary.reset_index().to_json(
    OUTPUT_DIR / "sbom_repository_summary.json",
    orient="records",
    indent=2
)

package_coverage.reset_index().to_json(
    OUTPUT_DIR / "sbom_shared_packages.json",
    orient="records",
    indent=2
)

print("Resultados exportados en:", OUTPUT_DIR.resolve())

## 24. Conclusión metodológica

El análisis de los SBOM de TanStack permite estudiar la **superficie de dependencias** de la organización desde la perspectiva de composición de software.

Este análisis no se limita a contar componentes. También identifica:

- repositorios con mayor cantidad de componentes;
- paquetes reutilizados por muchos repositorios;
- diversidad de versiones de una misma dependencia;
- componentes sin versión convencional;
- disponibilidad de identificadores PURL y CPE;
- cobertura de licencias;
- complejidad relativa de las relaciones de dependencia;
- componentes exclusivos y compartidos;
- concentración de la composición en determinados repositorios.

La interpretación conjunta recomendada para el proyecto es:

**CodeQL → problemas detectados en código fuente**  
**Syft/SBOM → inventario y composición de componentes**  
**Grype → vulnerabilidades conocidas asociadas a dependencias**

El SBOM funciona, por tanto, como el puente entre la composición del software y el análisis de vulnerabilidades de dependencias realizado por Grype.
